# 02 Exploration: Inspect, Validate and Profile the Dataset
**DSA 1080A | Project 06 | Group: 6**

Week 2 goal: inspect the raw data, check its quality and document problems **without changing the source data**. Cleaning happens in Week 3.

**How to run:** open this notebook from the `notebooks/` folder, then Run All from the top. Needs `pandas` and `openpyxl` (see `requirements.txt`).

## 1. Load the data

In [57]:
import pandas as pd
from pathlib import Path

local_path = Path("..") / "data" / "raw" / "digital_payment_transactions_dirty.xlsx"
url = "https://raw.githubusercontent.com/yuvraj56j-del/Digital-wallet-payment-and-consumer-choice-project/main/data/raw/digital_payment_transactions_dirty.xlsx"

df = pd.read_excel(local_path if local_path.exists() else url)
print("Loaded from:", local_path if local_path.exists() else "GitHub URL")
print("Shape:", df.shape)

Loaded from: GitHub URL
Shape: (18000, 10)


## 2. First look at the data

In [58]:
df.head(10)

,transaction_id,timestamp,merchant_category,amount,currency,payment_channel,customer_segment,region,transaction_status,day_of_week
0,TXN20250016413,2025-05-13 12:18:54,Cash Transfer,11070.29,KES,Mobile Wallet,Salaried Employee,Nairobi,Successful,Tuesday
1,TXN20250006473,2025-01-29 11:29:05,Government Services,3566.5,KES,QR Payment,Salaried Employee,Kiambu,Successful,Wednesday
2,TXN20250004783,2025-11-28 04:54:27,Shopping,4166.72,KES,NFC / Contactless,Salaried Employee,Nairobi,Successful,Friday
3,TXN20250009321,2025-12-05 14:13:05,Food & Dining,665.1,KES,in-app payment,Small Business Owner,Eldoret,Successful,Friday
4,TXN20250006513,2025-09-09 08:58:53,Travel,2794.76,KES,QR Payment,student,Nairobi,Successful,Tuesday
5,TXN20250014860,2025-04-29 06:28:40,NaN,756.34,KES,QR Payment,Other,Nairobi,Failed,Tues
6,TXN20250007135,2025-03-09 21:34:38,Food & Dining,906.73,KES,USSD,Other,Nakuru,Successful,Sunday
7,TXN20250016140,2025-08-31 08:53:02,Groceries,766.57,KES,USSD,Other,Kisumu,Successful,Sunday
8,TXN20250012179,2025-11-20 13:46:51,Food & Dining,77.06,KES,Mobile Wallet,Salaried Employee,Machakos,Successful,Thursday
9,TXN20250005859,2025-05-25 15:25:27,Transport,138.53,KES,Mobile Wallet,Salaried Employee,Mombasa,Successful,Sunday


In [59]:
df.tail(5)

,transaction_id,timestamp,merchant_category,amount,currency,payment_channel,customer_segment,region,transaction_status,day_of_week
17995,TXN20250004918,2025-01-04 11:48:13,Healthcare,3271.42,KES,Web Checkout,Salaried Employee,Mombasa,Successful,Saturday
17996,TXN20250011391,2025-09-07 17:52:26,Subscriptions,1104.06,KES,USSD,Student,Nairobi,Successful,Sunday
17997,TXN20250015158,2025-05-04 11:34:09,Healthcare,901.39,KES,Web Checkout,Salaried Employee,Nairobi,Successful,Sunday
17998,TXN20250001579,2025-04-25 17:22:07,Transport,1183.64,KES,Mobile Wallet,Salaried Employee,Kiambu,Failed,Friday
17999,TXN20250003092,2025-06-20 21:33:03,Utilities,1963.61,KES,In-App Payment,Other,Nairobi,Successful,Friday


In [60]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (18000, 10)


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   transaction_id      18000 non-null  str   
 1   timestamp           17938 non-null  str   
 2   merchant_category   17831 non-null  str   
 3   amount              17883 non-null  object
 4   currency            17986 non-null  str   
 5   payment_channel     17759 non-null  str   
 6   customer_segment    17730 non-null  str   
 7   region              17795 non-null  str   
 8   transaction_status  17892 non-null  str   
 9   day_of_week         17856 non-null  str   
dtypes: object(1), str(9)
memory usage: 3.0+ MB


In [62]:
df.dtypes

transaction_id           str
timestamp                str
merchant_category        str
amount                object
currency                 str
payment_channel          str
customer_segment         str
region                   str
transaction_status       str
day_of_week              str
dtype: object

In [63]:
df.describe(include="all")

,transaction_id,timestamp,merchant_category,amount,currency,payment_channel,customer_segment,region,transaction_status,day_of_week
count,18000,17938,17831,17883,17986,17759,17730,17795,17892,17856
unique,17965,17914,32,17314,5,24,21,25,13,28
top,TXN20250016734,2025-19-45,Groceries,-250,KES,Mobile Wallet,Salaried Employee,Nairobi,Successful,Sunday
freq,2,9,3071,15,17960,5991,5417,5552,16210,2575


**What this shows:** 18,000 rows and 10 columns. Almost every column is text. `amount` is stored as `object` (mixed text and numbers), which means some entries are not valid numbers. `timestamp` is text, not a real date type yet.

## 3. Data dictionary
One row per column (10 columns): name, type, meaning, valid values, expected blanks. "Stored type" is what Pandas read from the file; "Intended type" is what we convert to in Week 3. "Missing in raw file" is computed from the data. The same table is saved as `docs/data_dictionary.md`.

In [64]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

In [65]:
data_dictionary = pd.DataFrame({
    "Name": ["transaction_id", "timestamp", "merchant_category", "amount", "currency",
             "payment_channel", "customer_segment", "region", "transaction_status", "day_of_week"],
    "Stored type": df.dtypes.astype(str).values,
    "Intended type": ["text (ID)", "datetime", "category", "float", "category",
                      "category", "category", "category", "category", "category"],
    "Meaning": ["Transaction reference", "Date and time of the payment", "What the payment was for",
                "Transaction value", "Currency of the amount", "How the payment was made",
                "Type of customer", "Region of the transaction", "Outcome of the payment",
                "Weekday of the payment"],
    "Units / valid categories": [
        "TXN2025 + 7 digits (14 characters); should be unique",
        "YYYY-MM-DD HH:MM:SS, in 2025",
        "13: Groceries, Food & Dining, Transport, Shopping, Utilities, Entertainment, Fuel, Healthcare, Education, Cash Transfer, Subscriptions, Government Services, Travel",
        "KES, expected above zero",
        "KES",
        "6: Mobile Wallet, QR Payment, USSD, In-App Payment, NFC / Contactless, Web Checkout",
        "7: Salaried Employee, Student, Self-Employed, Small Business Owner, Gig Worker, Retired, Other",
        "10: Nairobi, Mombasa, Kisumu, Nakuru, Kiambu, Eldoret, Machakos, Nyeri, Kakamega, Other",
        "4: Successful, Failed, Pending, Reversed",
        "Monday to Sunday"],
    "Expected missing values": ["None", "None", "None", "None", "None", "None", "Few", "Few", "None",
                                "None (can be rebuilt from timestamp)"],
    "Missing in raw file": [f"{n} ({p:.2f}%)" for n, p in zip(df.isna().sum(), df.isna().mean() * 100)],
})

# Safety check: the typed column names must match the real columns, in the same order
assert list(df.columns) == data_dictionary["Name"].tolist()

data_dictionary

,Name,Stored type,Intended type,Meaning,Units / valid categories,Expected missing values,Missing in raw file
0,transaction_id,str,text (ID),Transaction reference,TXN2025 + 7 digits (14 characters); should be unique,None,0 (0.00%)
1,timestamp,str,datetime,Date and time of the payment,"YYYY-MM-DD HH:MM:SS, in 2025",None,62 (0.34%)
2,merchant_category,str,category,What the payment was for,"13: Groceries, Food & Dining, Transport, Shopping, Utilities, Entertainment, Fuel, Healthcare, Education, Cash Transfer, Subscriptions, Government Services, Travel",None,169 (0.94%)
3,amount,object,float,Transaction value,"KES, expected above zero",None,117 (0.65%)
4,currency,str,category,Currency of the amount,KES,None,14 (0.08%)
5,payment_channel,str,category,How the payment was made,"6: Mobile Wallet, QR Payment, USSD, In-App Payment, NFC / Contactless, Web Checkout",None,241 (1.34%)
6,customer_segment,str,category,Type of customer,"7: Salaried Employee, Student, Self-Employed, Small Business Owner, Gig Worker, Retired, Other",Few,270 (1.50%)
7,region,str,category,Region of the transaction,"10: Nairobi, Mombasa, Kisumu, Nakuru, Kiambu, Eldoret, Machakos, Nyeri, Kakamega, Other",Few,205 (1.14%)
8,transaction_status,str,category,Outcome of the payment,"4: Successful, Failed, Pending, Reversed",None,108 (0.60%)
9,day_of_week,str,category,Weekday of the payment,Monday to Sunday,None (can be rebuilt from timestamp),144 (0.80%)


**"None"** means the field should always be filled, so any blank is a data-quality issue. *"Few"* means a small number of blanks is plausible (the customer may not give the information)

## 4. Reusable functions
Two or more reusable functions are required. These are used again in Week 3 to compare before and after cleaning.

In [66]:
EXPECTED_COLUMNS = ["transaction_id", "timestamp", "merchant_category", "amount", "currency",
                    "payment_channel", "customer_segment", "region", "transaction_status", "day_of_week"]

def validate_columns(df, expected):
    """Check that the DataFrame has exactly the expected columns."""
    missing = [col for col in expected if col not in df.columns]
    extra = [col for col in df.columns if col not in expected]
    ok = len(missing) == 0 and len(extra) == 0
    return {"all_expected_present": ok, "missing": missing, "extra": extra}

def summarize_missingness(df):
    """Return null count and percentage for each column, largest first."""
    out = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_percent": (df.isna().mean() * 100).round(2),
    })
    return out.sort_values("null_count", ascending=False)

def summarize_categories(df, column):
    """Return every distinct label in a column with its count (shows spelling variants)."""
    return df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count")

print(validate_columns(df, EXPECTED_COLUMNS))

{'all_expected_present': True, 'missing': [], 'extra': []}


## 5. Missing values

In [67]:
missing = summarize_missingness(df)
missing

,null_count,null_percent
customer_segment,270,1.50
payment_channel,241,1.34
region,205,1.14
merchant_category,169,0.94
day_of_week,144,0.80
amount,117,0.65
transaction_status,108,0.60
timestamp,62,0.34
currency,14,0.08
transaction_id,0,0.00


In [68]:
print("Total blank cells:", int(missing["null_count"].sum()))
print("Rows with at least one blank:", int(df.isna().any(axis=1).sum()))

Total blank cells: 1330
Rows with at least one blank: 1300


## 6. Duplicates

In [69]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Repeated transaction_id values (extra copies):", df["transaction_id"].duplicated().sum())
dup_ids = df[df["transaction_id"].duplicated(keep=False)].sort_values("transaction_id")
dup_ids.head(6)

Fully duplicated rows: 0
Repeated transaction_id values (extra copies): 35


,transaction_id,timestamp,merchant_category,amount,currency,payment_channel,customer_segment,region,transaction_status,day_of_week
8077,TXN20250001709,2025-10-02 02:11:50,Shopping,2073.89,KES,QR Payment,Salaried Employee,Machakos,Successful,Thursday
11469,TXN20250001709,2025-04-04 10:41:16,Shopping,21970.14,KES,QR Payment,Self-Employed,Machakos,Successful,Friday
5257,TXN20250003037,2025-12-15 17:42:41,Government Services,914.34,KES,USSD,Gig Worker,Eldoret,Successful,Monday
5312,TXN20250003037,2025-09-07 12:09:53,Groceries,616.19,KES,QR Payment,Gig Worker,Eldoret,Successful,Sunday
1156,TXN20250003322,2025-03-08 01:06:07,Groceries,235.38,KES,NFC / Contactless,Self-Employed,Machakos,Reversed,Saturday
8240,TXN20250003322,2025-02-12 15:04:02,Shopping,971.49,KES,NFC / Contactless,Salaried Employee,Other,Failed,Wednesday


The repeated IDs have **different** timestamps and amounts, so they are ID clashes, not copied rows.

## 7. Invalid numeric ranges (`amount`)
We convert to numbers in a **separate variable** (`amount_num`); the original column is not changed.

In [70]:
amount_num = pd.to_numeric(df["amount"], errors="coerce")

not_numeric = df.loc[amount_num.isna() & df["amount"].notna(), "amount"]
print("Text entries in amount:", len(not_numeric))
print(not_numeric.value_counts())
print()
print("Negative amounts:", (amount_num < 0).sum())
print("Zero amounts:", (amount_num == 0).sum())
print("Amounts above 100,000 KES:", (amount_num > 100000).sum())
print("Amounts equal to 999,999,999:", (amount_num == 999999999).sum())
print()
print("Median amount:", round(amount_num.median(), 2))
print("Mean amount:", round(amount_num.mean(), 2))
amount_num.describe()

Text entries in amount: 14
amount
1,2OO      8
unknown    6
Name: count, dtype: int64

Negative amounts: 15
Zero amounts: 7
Amounts above 100,000 KES: 8
Amounts equal to 999,999,999: 5

Median amount: 1007.71
Mean amount: 281806.05


count    1.786900e+04
mean     2.818061e+05
std      1.672574e+07
min     -2.500000e+02
25%      4.796300e+02
50%      1.007710e+03
75%      2.140750e+03
max      1.000000e+09
Name: amount, dtype: float64

The mean is hugely larger than the median because of the extreme values, so we will report the **median** and flag outliers in Week 3.

In [71]:
sorted(amount_num.dropna())[-8:]

[153522.57,
 162956.02,
 180000.0,
 999999999.0,
 999999999.0,
 999999999.0,
 999999999.0,
 999999999.0]

## 8. Inconsistent categories

In [72]:
category_cols = ["merchant_category", "payment_channel", "customer_segment", "region", "transaction_status", "day_of_week", "currency"]
rows = []
for col in category_cols:
    s = df[col].dropna().astype(str)
    rows.append({"column": col,
                 "labels_in_file": s.nunique(),
                 "labels_after_strip_and_lowercase": s.str.strip().str.lower().nunique()})
pd.DataFrame(rows)

,column,labels_in_file,labels_after_strip_and_lowercase
0,merchant_category,32,21
1,payment_channel,24,13
2,customer_segment,21,13
3,region,25,15
4,transaction_status,13,9
5,day_of_week,28,21
6,currency,5,3


In [73]:
summarize_categories(df, 'merchant_category')

,merchant_category,count
0,Groceries,3071
1,Food & Dining,2742
2,Transport,2247
3,Shopping,2151
4,Utilities,1410
5,Entertainment,1092
6,Fuel,1037
7,Healthcare,891
8,Education,721
9,Cash Transfer,704


In [74]:
summarize_categories(df, 'transaction_status')

,transaction_status,count
0,Successful,16210
1,Failed,604
2,Pending,367
3,Reversed,357
4,successfull,121
5,Succesful,109
6,NaN,108
7,SUCCESSFUL,101
8,failed,6
9,pendng,4


Many labels are the same thing written differently (case, trailing spaces, typos such as `Groceris`, `successfull`, `Fryday`). Even after fixing case and spaces, typos remain, so a mapping will be needed in Week 3.

In [75]:
df["currency"].value_counts(dropna=False)

currency
KES     17960
NaN        14
kes        10
Ksh         7
USD         6
KES         3
Name: count, dtype: int64

## 9. Date parsing (`timestamp`)

In [76]:
ts = pd.to_datetime(df["timestamp"], errors="coerce")
unparsed = df.loc[ts.isna() & df["timestamp"].notna(), "timestamp"]

print("Blank timestamps:", df["timestamp"].isna().sum())
print("Non-blank but invalid timestamps:", len(unparsed))
print(unparsed.value_counts())
print()
print("Earliest valid date:", ts.min())
print("Latest valid date:", ts.max())

Blank timestamps: 62
Non-blank but invalid timestamps: 22
timestamp
2025-19-45          9
31/13/2025 25:61    7
not recorded        6
Name: count, dtype: int64

Earliest valid date: 2025-01-01 00:31:22
Latest valid date: 2025-12-31 23:45:05


## 10. Quality audit table

In [77]:
audit = pd.DataFrame([
    ["Missing values", "all columns except transaction_id", int(missing["null_count"].sum()), "Blank cells; highest in customer_segment and payment_channel"],
    ["Repeated IDs", "transaction_id", int(df["transaction_id"].duplicated().sum()), "Same ID, different details"],
    ["Fully duplicated rows", "all", int(df.duplicated().sum()), "None found"],
    ["Text in numeric column", "amount", len(not_numeric), "'unknown' and '1,2OO'"],
    ["Negative amounts", "amount", int((amount_num < 0).sum()), "Not valid for a payment"],
    ["Zero amounts", "amount", int((amount_num == 0).sum()), "Questionable"],
    ["Extreme amounts (999,999,999)", "amount", int((amount_num == 999999999).sum()), "Placeholder-like values"],
    ["Invalid timestamps", "timestamp", len(unparsed), "e.g. 31/13/2025 25:61"],
    ["Inconsistent category labels", "7 categorical columns", int(sum(r["labels_in_file"] - r["labels_after_strip_and_lowercase"] for r in rows)), "Extra labels from case/spacing variants (labels, not rows; typos not counted)"],
    ["Non-KES currency labels", "currency", int((~df["currency"].dropna().astype(str).str.strip().str.lower().isin(["kes"])).sum()), "USD and 'Ksh'"],
], columns=["Issue", "Column(s)", "Count", "Note"])
audit

,Issue,Column(s),Count,Note
0,Missing values,all columns except transaction_id,1330,Blank cells; highest in customer_segment and payment_channel
1,Repeated IDs,transaction_id,35,"Same ID, different details"
2,Fully duplicated rows,all,0,None found
3,Text in numeric column,amount,14,"'unknown' and '1,2OO'"
4,Negative amounts,amount,15,Not valid for a payment
5,Zero amounts,amount,7,Questionable
6,"Extreme amounts (999,999,999)",amount,5,Placeholder-like values
7,Invalid timestamps,timestamp,22,e.g. 31/13/2025 25:61
8,Inconsistent category labels,7 categorical columns,53,"Extra labels from case/spacing variants (labels, not rows; typos not counted)"
9,Non-KES currency labels,currency,13,USD and 'Ksh'


In [78]:
#for the 33.3% Mobile Wallet, 30.8% Nairobi, 90.1% Successful Observation
total = len(df)
for col, label in [("payment_channel", "Mobile Wallet"), ("region", "Nairobi"), ("transaction_status", "Successful")]:
    n = (df[col] == label).sum()
    print(f"{label}: {n} rows = {n / total * 100:.1f}% of all {total} rows")

Mobile Wallet: 5991 rows = 33.3% of all 18000 rows
Nairobi: 5552 rows = 30.8% of all 18000 rows
Successful: 16210 rows = 90.1% of all 18000 rows


In [80]:
from pathlib import Path

def to_markdown_table(table):
    """Turn a DataFrame into a Markdown table (no extra packages needed)."""
    clean = table.astype(str).apply(lambda col: col.str.replace("|", "/", regex=False))
    header = "| " + " | ".join(clean.columns) + " |"
    divider = "|" + "|".join(["---"] * len(clean.columns)) + "|"
    rows = ["| " + " | ".join(row) + " |" for row in clean.values]
    return "\n".join([header, divider] + rows)

Path("../docs").mkdir(exist_ok=True)
Path("../report").mkdir(exist_ok=True)

Path("../docs/data_dictionary.md").write_text(
    "# Data dictionary\n\n" + to_markdown_table(data_dictionary) + "\n", encoding="utf-8")
Path("../report/quality_audit.md").write_text(
    "# Quality audit table\n\n" + to_markdown_table(audit) + "\n", encoding="utf-8")

print("Saved: docs/data_dictionary.md and report/quality_audit.md")

Saved: docs/data_dictionary.md and report/quality_audit.md


# Initial observations
1. The dataset has **18,000 rows and 10 columns**(df.shape output), so it meets the 15,000 x 10 requirement.
2. Nine of the ten columns are text; `amount` is `object` because 14 entries are not valid numbers.(info()/dtypes and the text entries count in the invalid amounts section)
3. Every column except `transaction_id` has blanks (1,330 blank cells in total); `customer_segment` (1.50%) and `payment_channel` (1.34%) have the most.(from the missing values section)
4. The median amount is about KES 1,008, but the mean is about KES 281,806, pulled up by five values of 999,999,999.(from the invalid values section)
5. Before cleaning, Mobile Wallet is the largest channel (33.3% of rows), Nairobi the largest region (30.8%), and 90.1% of transactions are labelled Successful. These shares will change slightly after spelling variants are merged.


# Three potential data-quality issues
1. **Inconsistent category labels.** For example `merchant_category` has 32 labels in the file but only 13 intended categories; the rest are case, spacing or spelling variants. Counting categories without fixing this would split and understate groups.(from the category label table )
2. **Invalid and extreme amounts.** 14 text entries, 15 negative values, 7 zeros and 5 values of 999,999,999 would distort totals and averages.(from the invalid amounts section)
3. **Invalid timestamps and repeated IDs.** 22 timestamps cannot be parsed and 35 transaction IDs appear more than once with different details, which affects time-based analysis and any assumption that IDs are unique.(timestamps from the date-parsing section and 35 repeated IDs are from the duplicates section.)